# Remote setup (Google Colab)

Run each section in order. Inspect its output before continuing.
Project commands run in the locked uv environment. Colab retains its own kernel dependencies.
Use notebook 01 locally for interactive project imports.
Outside Colab, each remote operation is skipped.


## 1. Select the repository

Set the branch or tag before running the next section.
Use `chore/notebook-workflow` to review this PR. Use `main` after merge.


In [ ]:
import os
import shlex
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

from IPython import get_ipython

IN_COLAB = "google.colab" in sys.modules
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH_OR_REF = "main"
REPOSITORY_DIR = Path("/content/sleep-classifier")

if not IN_COLAB:
    print("Remote operations skipped outside Colab")

## 2. Open the checkout

Clone the selected branch if the directory does not exist.
Existing checkouts must match the selected local reference. Their files and branches remain unchanged.
The helper stops execution when a Git command fails.


In [ ]:
def run_checked(
    command: list[str | Path],
    *,
    cwd: Path | None = None,
    capture_output: bool = False,
) -> subprocess.CompletedProcess[str]:
    """Run a command and raise when it fails."""
    command_args = list(map(str, command))
    print("$ " + " ".join(command_args))
    try:
        result = subprocess.run(
            command_args,
            cwd=cwd,
            text=True,
            capture_output=capture_output,
            check=True,
        )
    except subprocess.CalledProcessError as error:
        if error.stdout:
            print(error.stdout, end="")
        if error.stderr:
            print(error.stderr, file=sys.stderr, end="")
        raise
    if capture_output:
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, file=sys.stderr, end="")
    return result


if IN_COLAB:
    if REPOSITORY_DIR.exists():
        if not (REPOSITORY_DIR / ".git").exists():
            raise RuntimeError(
                f"{REPOSITORY_DIR} exists and is not a Git checkout; "
                "move it manually before retrying."
            )
        current_commit = run_checked(
            ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, capture_output=True
        ).stdout.strip()
        selected_commit = run_checked(
            ["git", "rev-parse", "--verify", f"{BRANCH_OR_REF}^{{commit}}"],
            cwd=REPOSITORY_DIR,
            capture_output=True,
        ).stdout.strip()
        if current_commit != selected_commit:
            raise RuntimeError(
                "Existing checkout does not match BRANCH_OR_REF. Set the variable to its "
                "current branch or tag, or use another directory. No checkout changes were made."
            )
        status = run_checked(
            ["git", "status", "--short"], cwd=REPOSITORY_DIR, capture_output=True
        ).stdout
        print("Using existing checkout at", current_commit)
        if status:
            print("Existing checkout has local changes; they are preserved:\n" + status)
    else:
        REPOSITORY_DIR.parent.mkdir(parents=True, exist_ok=True)
        run_checked(["git", "clone", "--branch", BRANCH_OR_REF, REPOSITORY_URL, REPOSITORY_DIR])

    os.chdir(REPOSITORY_DIR)

## 3. Install uv

Install uv into a separate directory if necessary.
The shell prefix uses the Colab Python executable to locate uv.


In [ ]:
if IN_COLAB:
    UV_TARGET = Path("/content/.sleep-classifier-uv")
    if find_spec("uv") is None:
        run_checked(
            [sys.executable, "-m", "pip", "install", "--quiet", "--target", UV_TARGET, "uv"]
        )
        existing_pythonpath = os.environ.get("PYTHONPATH")
        os.environ["PYTHONPATH"] = str(UV_TARGET) + (
            os.pathsep + existing_pythonpath if existing_pythonpath else ""
        )

    UV = [sys.executable, "-m", "uv"]
    uv_command = shlex.join(UV)

## 4. Install the locked dependencies

Run the equivalent of `!uv sync --locked --group notebook`.
Shell failures raise an error before the next section.


In [ ]:
if IN_COLAB:
    !{uv_command} sync --locked --group notebook
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Dependency installation failed")

## 5. Download the participant

Run the download module. Existing files are reused only when their checksums match.


In [ ]:
if IN_COLAB:
    !{uv_command} run --locked --group notebook python -m sleep_classifier.data \
        download --config configs/explore.yaml
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Download failed")

## 6. Audit the participant

Run the audit module. Inspect the printed report before continuing.


In [ ]:
if IN_COLAB:
    !{uv_command} run --locked --group notebook python -m sleep_classifier.data \
        audit --config configs/explore.yaml
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Audit failed")

## 7. Execute notebook 01

Execute exploration in a fresh kernel from the uv environment.
The source notebook remains unchanged. A standalone HTML report is saved under `artifacts/notebooks`.


In [ ]:
if IN_COLAB:
    !{uv_command} run --locked --group notebook python scripts/execute_notebook.py
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Exploration report generation failed")


## 8. Display the report

Display the generated report below.


In [ ]:
if IN_COLAB:
    from IPython.display import HTML, display

    report_path = REPOSITORY_DIR / "artifacts/notebooks/01_data_exploration.executed.html"
    display(HTML(filename=str(report_path)))
